In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Many Bodies

Last time we ended with a question.  Everything lived on a line: one position $x$, one velocity $v$, and a potential that was a curve.  But a planet doesn't live on a line.  Its position has three components, and its potential energy depends on all of them.  So what does "minus the slope" mean when there are three directions you could walk in?

And a planet isn't alone, either.  The Earth pulls on the Moon and the Moon pulls on the Earth.  When there are lots of bodies, each one's potential energy depends on where all the *others* are.

Today we'll handle both: three dimensions, and many bodies.  The good news is that almost everything we've built keeps working.  The integrators won't change at all!

## Our Tools So Far

Here's what we need from last time, copied over without changes.  (Skim past them if you remember them!)

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE specifies the rate of change of a state y(t)
    in terms of
     - the state itself, y
     - the time, t.

    The generic form for an ODE is

    dy
    --  = f(t, y)
    dt

    '''

    def f(self, t, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, t, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')

In [ ]:
class SecondOrderODE(FirstOrderODE):
    '''
    A second-order ODE specifies the acceleration

    d^2x
    ----  = a(t, x, v)
    dt^2

    where v = dx/dt is the velocity.  Treating the state as y = (x, v) turns it into
    a first-order ODE,

    d  ( x )   (    v     )
    -- (   ) = (          )
    dt ( v )   ( a(t,x,v) )

    so a subclass only needs to provide the acceleration.
    '''

    def acceleration(self, t, x, v):
        raise NotImplementedError('The generic second-order ODE has an unspecified acceleration.')

    def f(self, t, y):
        # Unpack the state into its two components,
        x, v = y
        # and pack their rates of change back up the same way.
        return np.array([v, self.acceleration(t, x, v)])

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, t_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')

In [ ]:
class VelocityVerlet(FirstOrderODEIntegrator):
    '''
    Velocity Verlet splits each step into a half kick, a drift, and another half kick,

    v_{n+1/2} = v_n       + 1/2 a(t_n, x_n) Δt
    x_{n+1}   = x_n       + v_{n+1/2} Δt
    v_{n+1}   = v_{n+1/2} + 1/2 a(t_{n+1}, x_{n+1}) Δt

    It needs the ODE's acceleration, so it only works on second-order ODEs.
    '''

    def __init__(self, ODE, Delta_t, steps):

        self.Delta_t = Delta_t
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, t_0, y_0):

        # Populate all the t values
        t = t_0 + self.Delta_t * np.arange(self.steps+1)
        # and allocate an array with room for a whole state at each time.
        y = np.zeros((self.steps+1,) + np.shape(y_0))

        # The user picked a starting state
        y[0] = y_0

        for step in range(self.steps):
            x, v = y[step]
            # (acceleration also wants a velocity; we hand it the newest one we have.
            #  For forces that depend only on position it doesn't matter.)
            v_half = v      + 0.5 * self.ODE.acceleration(t[step],   x,     v)      * self.Delta_t   # half kick
            x_new  = x      +       v_half                                           * self.Delta_t   # drift
            v_new  = v_half + 0.5 * self.ODE.acceleration(t[step+1], x_new, v_half) * self.Delta_t   # half kick
            y[step+1] = np.array([x_new, v_new])

        return t, y

In [ ]:
class ConservativeSystem(SecondOrderODE):
    '''
    A conservative system has a potential energy V(x), and its acceleration is

    a = -V'(x)

    which doesn't depend on the velocity or the time.  Its energy

    E = 1/2 v^2 + V(x)

    is conserved.  A subclass provides the potential and the acceleration (minus its slope!).
    Everything is per unit mass.
    '''

    def potential(self, x):
        raise NotImplementedError('The generic conservative system has an unspecified potential.')

    def energy(self, x, v):
        return 0.5 * v**2 + self.potential(x)

# Minus the Slope in Every Direction

In three dimensions a body's position is a vector, $\vec{r} = (x, y, z)$, and its potential energy $V(\vec{r}) = V(x, y, z)$ depends on all three components.

Imagine walking on that landscape.  If you take a step in the $x$ direction, holding $y$ and $z$ fixed, the potential changes at the rate $\partial V / \partial x$.  That's a *partial derivative*: the ordinary slope, but only in the $x$ direction.  Last time the force was minus the slope, so the natural guess is that the $x$ component of the force is minus the slope in the $x$ direction, and the same for $y$ and $z$:
\begin{align}
    F_x &= -\frac{\partial V}{\partial x}
    &
    F_y &= -\frac{\partial V}{\partial y}
    &
    F_z &= -\frac{\partial V}{\partial z}.
\end{align}
That's exactly right.  (It comes from the same argument as last time: take $dE/dt$ and ask for it to vanish no matter which way the body is moving.)  We pack the three slopes into one vector called the *gradient*, $\nabla V$, and write
\begin{align}
    m \vec{a} &= \vec{F} = -\nabla V.
\end{align}
The gradient points uphill, in the direction where $V$ increases fastest, so the force points straight downhill.  A ball on a hillside rolls the steepest way down.

Notice the mass is back.  Last time we agreed to work per unit mass, because with one body the mass just goes along for the ride.  But soon we'll consider many bodies with *different* masses, like the Sun and the Earth, and there's no single mass we can divide out of everything.  So from now on each body has its own mass, and its acceleration is its force divided by its mass.

## Positions as Arrays

We'll store the position of a body as an array of 3 numbers, and the positions of $N$ bodies as an array of shape `(N, 3)`: one row per body, one column per direction.  The velocities have the same shape.

And of course we'll want to batch!  So the positions of a whole batch of systems have shape `(..., N, 3)`, where the `...` stands for however many batch axes we want, maybe none at all.  The two axes that belong to the system itself, the bodies and the directions, always come *last*.

That's a deliberate choice.  Broadcasting lines shapes up from the right, so as long as the system's axes come last, anything that only cares about bodies and directions will broadcast against any batch at all.  For example, if the masses `m` have shape `(N,)` then `m[:, np.newaxis]` has shape `(N, 1)`, and
```python
a = F / m[:, np.newaxis]
```
divides each body's force by its own mass, for any batch.

numpy even lets us write `...` in an index.  It means "all the axes I didn't mention".  So `x[..., 0, :]` is the position of body 0 in every system of the batch, however many batch axes there are, and `x[..., 2]` is the $z$ component of every body.

What about the whole state?  It's still a position and a velocity, so it has shape `(2, ..., N, 3)`.  Our integrators unpack it with `x, v = y`, which splits off the first axis, so they work without a single change.

## Energy Is a Sum

With many bodies the kinetic energy is a sum: each body contributes $\frac{1}{2} m_i |\vec{v}_i|^2$, and $|\vec{v}_i|^2 = v_{i,x}^2 + v_{i,y}^2 + v_{i,z}^2$.  So we have to sum over the bodies *and* over the directions,
\begin{align}
    E &= \sum_{i} \frac{1}{2} m_i |\vec{v}_i|^2 + V.
\end{align}
That means `ConservativeSystem`'s `energy`, $\frac{1}{2}v^2 + V$, isn't right anymore.  So let's make a new kind of conservative system that knows about masses, forces, and summing.  A specific system provides its `potential` and its `force`, and it gets `acceleration` and `energy` for free.

In [ ]:
class ManyBodySystem(ConservativeSystem):
    '''
    N bodies in three dimensions.  Positions and velocities have shape (..., N, 3),
    where the ... are any batch axes.  Body i has mass m[i], and

    m a = F = -∇V

    The energy is the kinetic energy of every body, plus the potential.
    A subclass provides the potential and the force (minus its gradient!).
    '''

    def __init__(self, masses):
        self.m = np.array(masses, dtype=float)

    def force(self, x):
        raise NotImplementedError('The generic many-body system has an unspecified force.')

    def acceleration(self, t, x, v):
        return self.force(x) / self.m[:, np.newaxis]

    def kinetic(self, v):
        # Sum over the bodies and the directions, but not over the batch.
        return np.sum(0.5 * self.m[:, np.newaxis] * v**2, axis=(-2, -1))

    def energy(self, x, v):
        return self.kinetic(v) + self.potential(x)

# One Body, Pulled Toward the Center

Let's start with one body ($N=1$) tied to the origin by a spring with spring constant $k$.  It's the harmonic oscillator, but in three dimensions:
\begin{align}
    V(\vec{r}) &= \frac{1}{2} k |\vec{r}|^2 = \frac{1}{2} k \left( x^2 + y^2 + z^2 \right).
\end{align}
The slope in the $x$ direction is $kx$, and the same for $y$ and $z$, so
\begin{align}
    \vec{F} &= -k \vec{r}.
\end{align}
The force always points straight at the origin.  A force like that is called *central*.

We can write it for any number of bodies, each tied to the origin by its own copy of the spring and ignoring the others.  Here we just need one.

In [ ]:
class CentralSpring(ManyBodySystem):
    '''
    Every body is tied to the origin by a spring with spring constant k, so

    V = sum_i 1/2 k |r_i|^2         F_i = -k r_i
    '''

    def __init__(self, masses, k):
        super().__init__(masses)
        self.k = k

    def potential(self, x):
        # summing over the last two axes, the axes that represent 
        # -2: which body we're talking about and
        # -1: which Cartesian coordinate we're talking about.
        return np.sum(0.5 * self.k * x**2, axis=(-2, -1))

    def force(self, x):
        return -self.k * x

## Trust, but Verify, in 3D

Last time we checked our calculus with a central difference.  The gradient is just three slopes, one per direction, so we can check each one the same way: nudge one coordinate up and down by $\Delta x$, holding everything else fixed.  With $N$ bodies there are $3N$ coordinates to nudge.

In [ ]:
def gradient(function, x, Delta_x=1e-5):
    '''
    Estimates the gradient of a function of positions x, with shape (..., N, 3),
    with a central difference in every coordinate of every body.
    '''
    N, d = np.shape(x)[-2:]
    result = np.zeros(np.shape(x))

    for i in range(N):
        for k in range(d):
            # Nudge only body i, in only direction k.
            step = np.zeros((N, d))
            step[i, k] = Delta_x
            result[..., i, k] = (function(x + step) - function(x - step)) / (2 * Delta_x)

    return result

def force_matches_potential(system, x):
    '''
    Checks whether a many-body system's force is minus the gradient of its potential.
    '''
    return np.allclose(system.force(x), -gradient(system.potential, x))

Notice that `step` has shape `(N, 3)`, so `x + step` nudges the same coordinate in every system of the batch at once.

Let's check at a batch of 10 random positions.  This should print `True`.

In [ ]:
rng = np.random.default_rng(320)

ball = CentralSpring([2.0], 8.0)

x = rng.normal(size=(10, 1, 3))
print(force_matches_potential(ball, x))

## Orbits

Now let's launch a batch of balls.  Each starts at a random place a distance 1 from the origin, with a random velocity.  With $m = 2$ and $k = 8$ the angular frequency is $\omega = \sqrt{k/m} = 2$, so a period is $\pi$.  Let's integrate for a few periods.

In [ ]:
orbits = 5

# Random directions, scaled to length 1,
r_0 = rng.normal(size=(orbits, 1, 3))
r_0 /= np.linalg.norm(r_0, axis=-1, keepdims=True)
# and random velocities.
v_0 = rng.normal(size=(orbits, 1, 3))

y_0 = np.array([r_0, v_0])
print(y_0.shape)

t, y = VelocityVerlet(ball, 0.01, 1000).starting_from(0, y_0)
print(y.shape)

The state starts with shape `(2, 5, 1, 3)`: position and velocity, 5 systems in the batch, 1 body, 3 directions.  The solution has one more axis in front, for time.

Let's look at the orbits in 3D.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection='3d')

for b in range(orbits):
    ax.plot(y[:, 0, b, 0, 0], y[:, 0, b, 0, 1], y[:, 0, b, 0, 2], color=f'C{b}')
    ax.scatter(*y[0, 0, b, 0], color=f'C{b}')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_aspect('equal')

Every orbit is a closed loop, and every one of them looks *flat*, tilted at its own angle.  Why should that be?

## Angular Momentum

A body's *angular momentum* about the origin is
\begin{align}
    \vec{L} &= m \, \vec{r} \times \vec{v}.
\end{align}
Let's see how it changes.  By the product rule,
\begin{align}
    \frac{d\vec{L}}{dt} &= m \, \vec{v} \times \vec{v} + \vec{r} \times m\vec{a} = \vec{r} \times \vec{F},
\end{align}
because the cross product of any vector with itself is zero.  For a central force, $\vec{F}$ points along $\vec{r}$, so $\vec{r} \times \vec{F} = 0$ too.  **A central force conserves angular momentum.**

That explains the flatness.  The cross product $\vec{r} \times \vec{v}$ is perpendicular to both $\vec{r}$ and $\vec{v}$.  If $\vec{L}$ never changes, then $\vec{r}$ is always perpendicular to one fixed direction, so the body stays in one plane forever.

`np.cross` takes the cross product along the last axis, so it broadcasts over the batch too.

In [ ]:
x, v = y[:, 0], y[:, 1]
L = np.sum(ball.m[:, np.newaxis] * np.cross(x, v), axis=-2)    # total over the bodies
print(L.shape)

print('largest change in L:', np.max(np.abs(L - L[0])))

The angular momentum doesn't change in any digit we can trust.  That's even better than the energy, which does wobble a little!  Velocity Verlet conserves angular momentum *exactly* for a central force, apart from roundoff.  The drift moves $\vec{r}$ along $\vec{v}$, which doesn't change $\vec{r} \times \vec{v}$, and each kick changes $\vec{v}$ along $\vec{r}$, which doesn't change it either.

Since each orbit lives in a plane, let's look at each one face-on.  We need two perpendicular directions in the plane.  The starting position $\vec{r}_0$ is in the plane, so it can be the first one.  The second has to be perpendicular to both $\vec{r}_0$ and $\vec{L}$, so we can get it with another cross product.

In [ ]:
def unit(vector):
    return vector / np.linalg.norm(vector, axis=-1, keepdims=True)

L_hat = unit(L[0])                      # (orbits, 3): perpendicular to each plane
e_1 = unit(r_0[:, 0])                   # (orbits, 3): in each plane
e_2 = np.cross(L_hat, e_1)              # (orbits, 3): in each plane, perpendicular to e_1

To see an orbit face-on we need its coordinates in the plane: how far it is along $\hat{e}_1$ and how far along $\hat{e}_2$.  Each one is a dot product, which for arrays is a product summed over the last axis.

In [ ]:
def project(vector, e_1, e_2):
    '''
    Projects vectors with shape (..., 3) onto the plane of the unit vectors e_1 and e_2,
    giving their two coordinates in that plane, with shape (..., 2).
    '''
    return np.stack([np.sum(vector * e_1, axis=-1), np.sum(vector * e_2, axis=-1)], axis=-1)

In [ ]:
r = x[:, :, 0]                          # (time, orbits, 3): the only body
print('largest distance out of the plane:', np.max(np.abs(np.sum(r * L_hat, axis=-1))))

face_on = project(r, e_1, e_2)          # (time, orbits, 2)
print(face_on.shape)

fig, ax = plt.subplots(figsize=(7, 7))
for b in range(orbits):
    ax.plot(face_on[:, b, 0], face_on[:, b, 1], color=f'C{b}')
ax.plot(0, 0, marker='+', color='black')
ax.set_aspect('equal')
ax.set_xlabel('along the starting position')
ax.set_ylabel('perpendicular, in the plane')

Face-on, every orbit is an ellipse centered on the origin.  That's what a 3D spring does: each component is its own harmonic oscillator, all with the same frequency.

(Gravity is a central force too, so planets conserve angular momentum and orbit in planes.  But gravity's orbits are ellipses with the Sun at a *focus*, not at the center.  You'll see that soon!)

## What About Phase Space?

In one dimension we got a lot out of phase space, with the position on one axis and the velocity on the other.  In three dimensions a single body needs three numbers for its position and three for its velocity, so its phase space is *six*-dimensional.  With $N$ bodies it's $6N$-dimensional.  That's exactly what our state array is: an array of shape `(2, N, 3)` holds $2 \times N \times 3 = 6N$ numbers, one point in phase space.  Good luck drawing that!

But the ideas carry over even when the pictures don't.  In the symplectic lesson we saw that velocity Verlet doesn't shrink or stretch areas in phase space.  In more dimensions the same is true of *volumes*: take a little blob of starting states, let every one of them evolve, and the blob may twist and stretch, but its six-dimensional (or $6N$-dimensional) volume stays the same.

# Many Bodies

Now for many bodies pulling on one another.  We'll connect them with springs.  Body $i$ and body $j$ are joined by a spring with spring constant $k_{ij}$, or $k_{ij} = 0$ if there's no spring between them.  It's the same spring either way, so $k_{ij} = k_{ji}$.  To keep things simple our springs have zero rest length: they pull harder the more they're stretched, and they pull all the way down to zero length.

The spring between $i$ and $j$ stores energy $\frac{1}{2} k_{ij} |\vec{r}_j - \vec{r}_i|^2$, so
\begin{align}
    V &= \sum_{i<j} \frac{1}{2} k_{ij} |\vec{r}_j - \vec{r}_i|^2.
\end{align}
We need $i<j$ so that we count every spring once.  Alternatively we can sum over *all* $i$ and $j$ and divide by 2, since every spring shows up twice, once as $(i,j)$ and once as $(j,i)$.  (The $i=j$ terms don't matter, because $\vec{r}_i - \vec{r}_i = 0$.)  That's easier with arrays:
\begin{align}
    V &= \frac{1}{4} \sum_{i, j} k_{ij} |\vec{r}_j - \vec{r}_i|^2.
\end{align}

Taking minus the gradient with respect to $\vec{r}_i$, the force on body $i$ is
\begin{align}
    \vec{F}_i &= \sum_j k_{ij} \left( \vec{r}_j - \vec{r}_i \right).
\end{align}
Each spring pulls body $i$ toward body $j$, harder the further apart they are.  And again the $j = i$ term is zero, so we don't have to skip it.

All of these sums need $\vec{r}_j - \vec{r}_i$ for every pair.  We've done that before!  In the broadcasting assignment we computed every displacement between 50 points in one line.  It's the same line here, with `...` in front for the batch.

In [ ]:
class SpringNetwork(ManyBodySystem):
    '''
    Bodies joined pairwise by zero-rest-length springs.  k[i, j] = k[j, i] is the spring
    constant between bodies i and j, or 0 if they aren't connected.

    V   = 1/2 sum_{i<j} k_ij |r_j - r_i|^2
    F_i =     sum_j     k_ij (r_j - r_i)
    '''

    def __init__(self, masses, k):
        super().__init__(masses)
        self.k = np.array(k, dtype=float)

    def displacement(self, x):
        # [..., i, j, :] is r_j - r_i, for every pair at once.
        return x[..., np.newaxis, :, :] - x[..., :, np.newaxis, :]

    def potential(self, x):
        stretch_squared = np.sum(self.displacement(x)**2, axis=-1)                 # (..., N, N)
        return 0.25 * np.sum(self.k * stretch_squared, axis=(-2, -1))

    def force(self, x):
        # Sum over j, the bodies pulling on body i.
        return np.sum(self.k[:, :, np.newaxis] * self.displacement(x), axis=-2)

Let's build a little 'molecule': four bodies with different masses, joined in a ring, $0 - 1 - 2 - 3 - 0$, plus one diagonal spring across from $0$ to $2$.

(Note that this isn't a realistic model of a molecule.  For one thing chemical bonds are not springs: if you pull atoms far enough apart the bonds yield but springs do not.  But it's useful language!)

In [ ]:
masses = [1.0, 2.0, 3.0, 4.0]
k = np.array([
    [0, 1, 1, 1],
    [1, 0, 1, 0],
    [1, 1, 0, 1],
    [1, 0, 1, 0],
])

molecule = SpringNetwork(masses, k)

x = rng.normal(size=(10, 4, 3))
print(force_matches_potential(molecule, x))

Our calculus checks out.  Now let's give a batch of molecules random starting positions and velocities, and integrate.

In [ ]:
molecules = 3

x_0 = rng.normal(size=(molecules, 4, 3))
v_0 = rng.normal(size=(molecules, 4, 3))

t, y = VelocityVerlet(molecule, 0.01, 2000).starting_from(0, np.array([x_0, v_0]))
x, v = y[:, 0], y[:, 1]
print(y.shape)

Here's what the first molecule looks like at the start.  Each dot is a body, bigger for heavier ones, and each gray line is a spring, one for every nonzero $k_{ij}$.

In [ ]:
# Every connected pair, counted once
pairs = [(i, j) for i in range(len(masses)) for j in range(i + 1, len(masses)) if k[i, j] != 0]

def draw_molecule(ax, positions):
    '''
    Draws the springs and the bodies of one molecule, with positions of shape (N, 3).
    Returns the lines it drew, so that they can be moved later.
    '''
    springs = [ax.plot(*positions[[i, j]].T, color='gray')[0] for i, j in pairs]
    bodies = [ax.plot(*positions[i, :, np.newaxis], color=f'C{i}', marker='o', markersize=8 * np.sqrt(mass), linestyle='none')[0]
              for i, mass in enumerate(masses)]
    return springs, bodies

start = x[0, 0]                         # (N, 3): molecule 0 at t = 0

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection='3d')
ax.set_proj_type('ortho')               # no perspective, so near things don't look bigger than far ones
draw_molecule(ax, start)

# A cube around the molecule, so that no direction gets squashed
middle = np.mean(start, axis=0)
half = np.max(np.abs(start - middle))
ax.set_xlim(middle[0] - half, middle[0] + half)
ax.set_ylim(middle[1] - half, middle[1] + half)
ax.set_zlim(middle[2] - half, middle[2] + half)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_aspect('equal')

## Momentum and the Center of Mass

Notice something about the force.  Body $j$ pulls on body $i$ with $k_{ij}(\vec{r}_j - \vec{r}_i)$, and body $i$ pulls on body $j$ with $k_{ji}(\vec{r}_i - \vec{r}_j)$.  That's the same size, in the opposite direction: Newton's third law!  So when we add up all the forces, every pair cancels,
\begin{align}
    \sum_i \vec{F}_i &= 0.
\end{align}
The total momentum $\vec{P} = \sum_i m_i \vec{v}_i$ changes at the rate $\sum_i m_i \vec{a}_i = \sum_i \vec{F}_i = 0$, so it's conserved.

That means the *center of mass*
\begin{align}
    \vec{R} &= \frac{1}{M} \sum_i m_i \vec{r}_i
    &
    M &= \sum_i m_i
\end{align}
moves at the constant velocity $\vec{P}/M$, in a straight line.  However wildly the bodies fling each other around, the molecule as a whole just coasts.

Let's check energy, momentum, and the center of mass, for every molecule in the batch.

In [ ]:
M = np.sum(molecule.m)

energy = molecule.energy(x, v)                                          # (time, molecules)
P = np.sum(molecule.m[:, np.newaxis] * v, axis=-2)                      # (time, molecules, 3)
R = np.sum(molecule.m[:, np.newaxis] * x, axis=-2) / M                  # (time, molecules, 3)

# Where the center of mass should be, moving in a straight line at constant velocity.
coasting = R[0] + t[:, np.newaxis, np.newaxis] * P[0] / M

print('largest change in E / E(0):', np.max(np.abs(energy / energy[0] - 1), axis=0))
print('largest change in P:       ', np.max(np.abs(P - P[0]), axis=(0, -1)))
print('largest distance from straight-line coasting:', np.max(np.abs(R - coasting), axis=(0, -1)))

The energy wobbles a tiny bit, the way it always does with velocity Verlet.  But the momentum doesn't change at all, apart from roundoff.  That's the same story as the angular momentum: each kick adds up to zero total change in momentum, because the forces cancel in pairs.

Let's watch one molecule.  The thick black line is the center of mass.

In [ ]:
b = 0

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(projection='3d')

for i in range(len(masses)):
    ax.plot(x[:, b, i, 0], x[:, b, i, 1], x[:, b, i, 2], color=f'C{i}', alpha=0.6, label=f'm = {masses[i]}')
ax.plot(R[:, b, 0], R[:, b, 1], R[:, b, 2], color='black', linewidth=3, label='center of mass')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_aspect('equal')
ax.legend()

The bodies swirl around each other in three dimensions while the center of mass glides along in a perfectly straight line.

The drift makes it hard to see what the springs are doing.  So let's ride along with the center of mass: subtract $\vec{R}$ from every body's position, and the 'atoms' in the molecule sit around the origin.

In [ ]:
# Positions relative to the center of mass, for every molecule: (time, molecules, N, 3)
relative = x - R[..., np.newaxis, :]

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(projection='3d')

for i in range(len(masses)):
    ax.plot(relative[:, b, i, 0], relative[:, b, i, 1], relative[:, b, i, 2], color=f'C{i}', alpha=0.6, label=f'm = {masses[i]}')

# A cube around the center of mass, so that no direction gets squashed
limit = np.max(np.abs(relative[:, b]))
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_zlim(-limit, limit)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_aspect('equal')
ax.set_title('Motion relative to the center of mass')
ax.legend()

We can watch the molecule wiggle.  If $t$ is in seconds, the movie plays in real time.

You may need to run `uv sync` in a terminal from the root directory of this repository for this to work.

In [ ]:
import imageio_ffmpeg
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Point matplotlib at the copy of ffmpeg that came with `uv sync`.
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection='3d')
springs, bodies = draw_molecule(ax, relative[0, b])
ax.plot(0, 0, 0, marker='+', color='black')

limit = np.max(np.abs(relative[:, b]))
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_zlim(-limit, limit)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
clock = ax.set_title('')

def draw(step):
    for spring, (i, j) in zip(springs, pairs):
        spring.set_data_3d(*relative[step, b, [i, j]].T)
    for i, body in enumerate(bodies):
        body.set_data_3d(*relative[step, b, i, :, np.newaxis])
    clock.set_text(f't = {t[step]:.2f}')

# Show every frame_step-th step: 4 steps of 0.01 is 0.04 per frame, 25 frames per second.
frame_step = 4
animation = FuncAnimation(fig, draw, frames=range(0, len(t), frame_step), interval=1000 * frame_step * (t[1] - t[0]))
plt.close(fig)   # show only the movie, not a leftover still frame
HTML(animation.to_html5_video())

# What About Gravity?

Today we learned to

 - get the force from the potential in 3D, $\vec{F} = -\nabla V$,
 - check our calculus one coordinate at a time,
 - store $N$ bodies in an array of shape `(..., N, 3)` and batch over many systems,
 - compute every pair's displacement at once, and
 - check conservation of energy, momentum, and angular momentum.

Gravity is a pair force too, with
\begin{align}
    V &= -\sum_{i<j} \frac{G m_i m_j}{|\vec{r}_j - \vec{r}_i|}.
\end{align}
It's not much different from our springs.  But look at the denominator.  Our springs got away with summing over every $i$ and $j$, because the $i = j$ terms were zero.  Gravity's $i = j$ terms would divide by zero!  So we can't keep those pieces on the body diagonal---we'll have to get rid of them.